# LightRAG 10-K Test Pipeline - Optimized Models & Multimode Query

**Goal**: Test optimized LightRAG with multiple query modes and improved models

**Test Strategy**:
1. Index CTAS (Cintas) document only for initial testing
2. Query with all CTAS questions from finder_train.parquet
3. Test all 5 LightRAG query modes: local, global, naive, hybrid, mixed
4. Generate separate output files for each mode
5. If successful, uncomment to batch index all 10 priority tickers

**Optimized Models**:
- Embedding: intfloat/e5-mistral-7b-instruct (4096 dim, GPU)
- Entity Extraction LLM: ministral-14b-2512 (Mistral API)
- Generation LLM: gemini-3-flash-preview (Google Gemini API)
- Reranker: BAAI/bge-reranker-v2-m3 (8192 context, GPU)
- Chunking: 1200 tokens, 100 overlap

**Priority Tickers**: HON, MSI, MRO, CE, HAS, VRSK, BIIB, ROP, CTAS, AMP

In [ ]:
# Standard library
import os
import json
import asyncio
from pathlib import Path
from functools import partial
from datetime import datetime
import re

# Third-party
from dotenv import load_dotenv
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import nest_asyncio

# LightRAG
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

# ML Models
from sentence_transformers import SentenceTransformer, CrossEncoder

# Google Gemini
import google.genai as genai

# Enable nested async for Jupyter
nest_asyncio.apply()
load_dotenv()

# ============================================
# CUDA Memory Optimization
# ============================================
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print("⚡ CUDA memory fragmentation fix enabled\n")

# ============================================
# Configuration
# ============================================

# Paths
PARSED_DOCS_PATH = "parsed_10k_documents.json"
WORKING_DIR = "./lightrag_10k_workspace_latest"
TEST_QUESTIONS_PATH = "finder_train.parquet"
RESULTS_BASE_PATH = "test_results_CTAS"  # Will append mode name

# Model names
EMBEDDING_MODEL_NAME = "intfloat/e5-mistral-7b-instruct"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
MISTRAL_MODEL = "ministral-14b-2512"
GEMINI_MODEL = "gemini-3-flash-preview"
EMBEDDING_DIM = 4096  # e5-mistral-7b-instruct dimension

# GPU setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("="*60)
print(f"🚀 Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("="*60 + "\n")

# Priority tickers
PRIORITY_TICKERS = ["HON", "MSI", "MRO", "CE", "HAS", "VRSK", "BIIB", "ROP", "CTAS", "AMP"]

# Query modes to test
QUERY_MODES = ["local", "global", "naive", "hybrid", "mix"]

print("✓ Configuration loaded")

In [ ]:
print("Loading models...\n")

# ============================================
# AGGRESSIVE GPU MEMORY CLEANUP
# ============================================
import gc

# Delete any existing model variables from previous runs
if 'embedding_model' in globals():
    del embedding_model
    print("  Deleted previous embedding_model")

if 'reranker' in globals():
    del reranker
    print("  Deleted previous reranker")

if 'rag' in globals():
    del rag
    print("  Deleted previous rag instance")

# Force garbage collection
gc.collect()

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    free_memory = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {free_memory:.2f}GB free / {total_memory:.2f}GB total\n")
else:
    print("  CPU mode - no GPU available\n")

gc.collect()

# ============================================
# 1. E5-Mistral-7B Embedding Model (4-bit Quantization)
# ============================================
print(f"Loading {EMBEDDING_MODEL_NAME} with 4-bit quantization...")
print("  4-bit quantization reduces VRAM from 13GB → 3.4GB")
print("  This will easily fit on your GPU!")
print()

from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("  [1/3] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

print("  [2/3] Loading model with 4-bit quantization (this may take 30-60 seconds)...")
base_model = AutoModel.from_pretrained(
    EMBEDDING_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("  [3/3] Wrapping in SentenceTransformer interface...")

# Create a custom SentenceTransformer wrapper for the quantized model
class QuantizedE5Model:
    def __init__(self, model, tokenizer, embedding_dim):
        self.model = model
        self.tokenizer = tokenizer
        self.embedding_dim = embedding_dim
        self.device = next(model.parameters()).device
        
    def encode(self, sentences, normalize_embeddings=True, batch_size=4, **kwargs):
        """Encode sentences to embeddings"""
        if isinstance(sentences, str):
            sentences = [sentences]
        
        all_embeddings = []
        
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i:i+batch_size]
            
            # Tokenize
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            ).to(self.device)
            
            # Get embeddings
            with torch.no_grad():
                outputs = self.model(**encoded)
                # Mean pooling
                embeddings = outputs.last_hidden_state.mean(dim=1)
                
                if normalize_embeddings:
                    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
                
                all_embeddings.append(embeddings.cpu().float().numpy())
        
        return np.vstack(all_embeddings)

embedding_model = QuantizedE5Model(base_model, tokenizer, EMBEDDING_DIM)
EMBEDDING_DEVICE = "cuda"

# Show VRAM usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
    
    print(f"\n✓ Embedding model loaded (4-bit quantized)")
    print(f"  Dimension: {EMBEDDING_DIM}")
    print(f"  Device: {EMBEDDING_DEVICE}")
    print(f"  VRAM: {allocated:.2f}GB allocated / {reserved:.2f}GB reserved / {free:.2f}GB free\n")

# Test embedding
print("  Running test embedding...")
test_embed = embedding_model.encode(["test"], normalize_embeddings=True, batch_size=1)
assert test_embed.shape[1] == EMBEDDING_DIM, f"Expected {EMBEDDING_DIM} dims, got {test_embed.shape[1]}"
print(f"  Test embedding: {test_embed.shape} ✓\n")

# Clear cache after test
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# ============================================
# 2. Mistral 14B LLM for Entity Extraction (API)
# ============================================
async def mistral_extraction_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    """Mistral 14B LLM wrapper for entity extraction"""
    return await openai_complete_if_cache(
        MISTRAL_MODEL,
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("MISTRAL_API_KEY"),
        base_url="https://api.mistral.ai/v1",
        **kwargs
    )

print(f"✓ Entity Extraction LLM configured: {MISTRAL_MODEL}\n")

# ============================================
# 3. Google Gemini for Generation (API)
# ============================================
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

async def gemini_generation_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    """Google Gemini wrapper for answer generation"""
    try:
        # Build full prompt with system prompt and history
        full_prompt = ""
        if system_prompt:
            full_prompt += f"{system_prompt}\n\n"
        
        if history_messages:
            for msg in history_messages:
                role = msg.get("role", "user")
                content = msg.get("content", "")
                full_prompt += f"{role}: {content}\n\n"
        
        full_prompt += prompt
        
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=full_prompt
        )
        
        return response.text
    
    except Exception as e:
        print(f"Gemini API Error: {e}")
        return f"Error generating response: {str(e)}"

print(f"✓ Generation LLM configured: {GEMINI_MODEL}\n")

# ============================================
# 4. E5-Mistral Embedding Function (Async Wrapper)
# ============================================
async def e5_embedding_func(texts: list[str]) -> np.ndarray:
    """Async wrapper for E5-Mistral embeddings"""
    return embedding_model.encode(
        texts, 
        normalize_embeddings=True, 
        batch_size=4
    )

print("✓ Embedding function wrapped\n")

# ============================================
# 5. Reranker Setup (Deferred - will load later)
# ============================================
print("⏳ Reranker: Will load before querying (deferred to save VRAM during indexing)")
reranker = None  # Will be loaded in Cell 6 before querying

async def hf_rerank_func(query: str, documents: list[str], top_n: int = 20, **kwargs):
    """Placeholder - will be defined when reranker is loaded"""
    global reranker
    if reranker is None:
        raise RuntimeError("Reranker not loaded yet. Load it before querying.")
    
    pairs = [[query, doc] for doc in documents]
    scores = reranker.predict(pairs, show_progress_bar=False, batch_size=8)
    results = [{"index": i, "relevance_score": float(score)} for i, score in enumerate(scores)]
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:top_n]

print("="*60)
print("✓ Core models loaded successfully")
print(f"  E5-Mistral: CUDA 4-bit (embeddings)")
print("  Reranker: Deferred (will load before querying)")
print("="*60)

In [ ]:
print("\nInitializing LightRAG...\n")

# Create workspace
Path(WORKING_DIR).mkdir(exist_ok=True)

# ============================================
# Initialize LightRAG Instance
# ============================================
rag = LightRAG(
    working_dir=WORKING_DIR,
    
    # LLM - Use Mistral 14B for entity extraction
    llm_model_func=mistral_extraction_func,
    llm_model_name=MISTRAL_MODEL,
    
    # Embeddings - E5-Mistral-7B with 4096 dimensions
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBEDDING_DIM,
        max_token_size=8192,
        func=e5_embedding_func
    ),
    
    # Chunking
    chunk_token_size=1200,
    chunk_overlap_token_size=100,
    
    # Reranker - BGE v2-m3 with 8K context
    rerank_model_func=hf_rerank_func,
    min_rerank_score=0.3
)

await rag.initialize_storages()

print("✓ LightRAG initialized")
print(f"  Working dir: {WORKING_DIR}")
print(f"  Chunk size: 1200 tokens, 100 overlap")
print(f"  Entity extraction: {MISTRAL_MODEL}")
print(f"  Embedding: {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIM}D)")
print(f"  Reranker: {RERANKER_MODEL_NAME} (8K context, GPU)\n")

# ============================================
# Load Documents
# ============================================
with open(PARSED_DOCS_PATH, 'r') as f:
    documents = json.load(f)

print(f"✓ Loaded {len(documents)} company documents\n")

# ============================================
# Index Function with Metadata
# ============================================
async def index_document(ticker: str, doc_data: dict):
    """Index document with full metadata encoded in file_path"""
    
    # Encode metadata as compact JSON
    metadata_json = json.dumps({
        "ticker": ticker,
        "company": doc_data.get("company_name", ""),
        "period": doc_data.get("period_end_date", ""),
        "source": doc_data.get("source_file", "")
    }, separators=(',', ':'))
    
    # Insert document (LightRAG chunks automatically)
    await rag.ainsert(
        input=doc_data["text"],
        ids=ticker,
        file_paths=metadata_json
    )

In [ ]:
# ============================================
# TEST: Index CTAS Only
# ============================================
print("="*60)
print("📍 TEST: Indexing CTAS (Cintas Corporation) only")
print("="*60)

TEST_TICKER = "CTAS"

if TEST_TICKER not in documents:
    print(f"❌ ERROR: {TEST_TICKER} not found in documents!")
else:
    print(f"\nCompany: {documents[TEST_TICKER]['company_name']}")
    print(f"Period: {documents[TEST_TICKER]['period_end_date']}")
    print(f"Text length: {len(documents[TEST_TICKER]['text']):,} chars\n")
    
    print(f"Indexing {TEST_TICKER}... (this may take a few minutes)")
    await index_document(TEST_TICKER, documents[TEST_TICKER])
    
    print(f"\n✓ {TEST_TICKER} indexed successfully!")
    print("  - LightRAG chunked document automatically")
    print("  - Built knowledge graph entities/relations")
    print("  - Stored metadata for citation/provenance")

print("\n" + "="*60)
print("Ready for querying!")
print("="*60)

# ============================================
# FOR FULL PIPELINE: Index All 10 Priority Tickers
# Uncomment below after successful CTAS test
# ============================================
'''
print("\n\n📍 Indexing all 10 priority tickers...")
BATCH_PRIORITY = [t for t in PRIORITY_TICKERS if t != "" and t in documents]
for ticker in tqdm(BATCH_PRIORITY, desc="Priority batch"):
    await index_document(ticker, documents[ticker])

print(f"\n✓ Indexed {len(BATCH_PRIORITY)} priority tickers")
'''

In [ ]:
print("\nLoading test questions...\n")

# ============================================
# Helper Functions
# ============================================

def to_python_type(val):
    """Convert pandas/numpy types to native Python types for JSON serialization"""
    if pd.isna(val):
        return None
    elif isinstance(val, (np.integer, np.int64, np.int32)):
        return int(val)
    elif isinstance(val, (np.floating, np.float64, np.float32)):
        return float(val)
    elif isinstance(val, np.bool_):
        return bool(val)
    elif isinstance(val, np.ndarray):
        return val.tolist()
    elif isinstance(val, (list, tuple)):
        return str(val)  # Convert lists to string for JSON
    else:
        return val

def extract_tickers_from_text(text: str) -> list:
    """Extract ticker symbols from text"""
    tickers = []
    
    # Pattern 1: Ticker in parentheses
    paren_tickers = re.findall(r'\(([A-Z]{1,5})\)', text)
    tickers.extend(paren_tickers)
    
    # Pattern 2: Ticker at end with dash or comma
    end_tickers = re.findall(r'[,\-]\s*([A-Z]{1,5})\s*[,.]?\s*$', text)
    tickers.extend(end_tickers)
    
    return list(set(t for t in tickers if len(t) <= 5))

# ============================================
# Load and Filter Questions for CTAS
# ============================================
df_all = pd.read_parquet(TEST_QUESTIONS_PATH)
print(f"✓ Loaded {len(df_all)} total questions\n")

# Filter for CTAS questions only
def has_ctas(text):
    tickers = extract_tickers_from_text(text)
    return TEST_TICKER in tickers

ctas_mask = df_all['text'].apply(has_ctas)
test_questions = df_all[ctas_mask]

print(f"✓ Found {len(test_questions)} questions for {TEST_TICKER}\n")

# ============================================
# FOR FULL PIPELINE: Filter for All Priority Tickers
# Uncomment below after successful CTAS test
# ============================================
'''
def filter_priority_questions(df: pd.DataFrame) -> pd.DataFrame:
    """Filter questions for all priority tickers"""
    def has_priority_ticker(text):
        tickers = extract_tickers_from_text(text)
        return any(ticker in PRIORITY_TICKERS for ticker in tickers)
    
    mask = df['text'].apply(has_priority_ticker)
    return df[mask]

test_questions = filter_priority_questions(df_all)
print(f"✓ Found {len(test_questions)} questions for all priority tickers\n")

# Count per ticker
print("Questions per ticker:")
for ticker in PRIORITY_TICKERS:
    count = test_questions['text'].apply(
        lambda text: ticker in extract_tickers_from_text(text)
    ).sum()
    print(f"  {ticker}: {count} questions")
print(f"\n  TOTAL: {len(test_questions)} questions\n")
'''

In [ ]:
# ============================================
# Load Reranker Now (Before Querying)
# ============================================

if reranker is None:
    print("="*60)
    print("Loading reranker for querying...")
    print("="*60)
    print()
    
    # Decide GPU vs CPU based on available VRAM
    if DEVICE == "cuda":
        free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
        print(f"Available VRAM: {free_vram:.2f}GB")
        
        if free_vram > 2.0:
            print("✓ Sufficient VRAM - loading reranker on GPU (FP16)")
            reranker_device = "cuda"
            use_fp16 = True
        else:
            print("⚠ Low VRAM - loading reranker on CPU to avoid OOM")
            reranker_device = "cpu"
            use_fp16 = False
    else:
        reranker_device = "cpu"
        use_fp16 = False
    
    reranker = CrossEncoder(
        RERANKER_MODEL_NAME,
        device=reranker_device,
        max_length=8192
    )
    
    if use_fp16:
        reranker.model.half()
        reranker.model.eval()
    
    print(f"✓ Reranker loaded on {reranker_device}")
    
    if DEVICE == "cuda":
        allocated = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1e9
        print(f"  VRAM: {allocated:.2f}GB allocated / {reserved:.2f}GB reserved / {free:.2f}GB free")
    
    print()

# ============================================
# Query Function for Single Mode
# ============================================

async def query_single_mode(question_text: str, query_mode: str):
    """Query LightRAG with specified mode and return results"""
    
    try:
        # Query with specified mode + reranking
        context = await rag.aquery(
            question_text,
            param=QueryParam(
                mode=query_mode,
                top_k=60,
                chunk_top_k=10,
                only_need_context=True,
                enable_rerank=True
            )
        )
        
        return {
            "retrieved_context": context,
            "context_length": len(context),
            "timestamp": datetime.now().isoformat(),
            "status": "success"
        }
    
    except Exception as e:
        return {
            "retrieved_context": None,
            "context_length": 0,
            "error": str(e),
            "timestamp": datetime.now().isoformat(),
            "status": "error"
        }

# ============================================
# Run Queries: One File Per Question
# ============================================

print("="*60)
print(f"🚀 Processing {len(test_questions)} questions across {len(QUERY_MODES)} modes")
print("   Output: One file per question with all mode results")
print("="*60)
print()

total_questions = len(test_questions)

# Iterate through each question
for q_idx, (_, row) in enumerate(test_questions.iterrows(), 1):
    question_id = str(row["_id"])
    question_text = str(row["text"])
    
    print(f"\n{'='*60}")
    print(f"[{q_idx}/{total_questions}] Question ID: {question_id}")
    print(f"Question: {question_text[:70]}...")
    print("="*60)
    
    # Collect results for all modes
    modes_results = {}
    
    # Query this question with all 5 modes
    for mode_idx, mode in enumerate(QUERY_MODES, 1):
        print(f"  [{mode_idx}/{len(QUERY_MODES)}] Querying with {mode.upper()} mode...", end=" ")
        
        mode_result = await query_single_mode(question_text, mode)
        modes_results[mode] = mode_result
        
        if mode_result["status"] == "success":
            print(f"✓ {mode_result['context_length']:,} chars")
        else:
            print(f"❌ Error: {mode_result.get('error', 'Unknown')[:50]}")
    
    # Build complete question result
    question_result = {
        "question_id": question_id,
        "question": question_text,
        "expected_answer": to_python_type(row["answer"]),
        "expected_type": to_python_type(row.get("type")),
        "category": to_python_type(row.get("category")),
        "reasoning": to_python_type(row.get("reasoning")),
        "references": to_python_type(row.get("references")),
        "modes": modes_results
    }
    
    # Save to individual file
    output_path = f"{RESULTS_BASE_PATH}_question_{question_id}.json"
    with open(output_path, 'w') as f:
        json.dump(question_result, f, indent=2, ensure_ascii=False)
    
    print(f"  💾 Saved: {output_path}")
    
    # Summary for this question
    success_modes = [m for m, r in modes_results.items() if r["status"] == "success"]
    error_modes = [m for m, r in modes_results.items() if r["status"] == "error"]
    
    if success_modes:
        avg_context = np.mean([modes_results[m]['context_length'] for m in success_modes])
        print(f"  ✓ Success: {len(success_modes)}/{len(QUERY_MODES)} modes | Avg context: {avg_context:,.0f} chars")
    
    if error_modes:
        print(f"  ⚠ Errors in modes: {', '.join(error_modes)}")

# ============================================
# Final Summary
# ============================================

print("\n" + "="*60)
print("🎉 ALL QUESTIONS PROCESSED!")
print("="*60)
print()
print(f"Total files generated: {total_questions}")
print(f"File pattern: {RESULTS_BASE_PATH}_question_{{question_id}}.json")
print()
print("Each file contains:")
print("  - Question metadata (ID, text, expected answer, type, category)")
print("  - Results from all 5 query modes (local, global, naive, hybrid, mixed)")
print()
print("Next steps:")
print("  1. Analyze mode comparison within each question file")
print("  2. Generate answers using Gemini (separate script)")
print("  3. Evaluate answer quality across modes")
print("  4. If successful, uncomment and run full 10-ticker pipeline")
print("="*60)